In [109]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 50)

from dataset import Dataset


# ---------------------------------------------------------------
# 0.1 Structural sanity check
# ---------------------------------------------------------------
def structural_check(df, experiment_col="experiment_id", fiber_col="fiber_id"):
    print("=== 0.1 Structural check ===")
    print(f"Rows (fibers): {len(df)}")
    print(f"Unique experiments: {df[experiment_col].nunique()}")
    if fiber_col in df.columns:
        print(f"Duplicate fiber IDs: {df[fiber_col].duplicated().sum()}")
        print("\nColumn dtypes:\n", df.dtypes.value_counts())
        print("\n>> Expect ~367 experiments, ~2190 fibers. If these don't match,")
        print("\n>> Expect ~367 experiments, ~2190 fibers. If these don't match,")
        print(" check for duplicated rows or a schema mismatch before")
        print(" proceeding — every downstream diagnostic assumes this is right.\n")

# ---------------------------------------------------------------
# 0.2 Fibers-per-experiment distribution
# ---------------------------------------------------------------
def fiber_count_distribution(df, experiment_col="experiment_id"):
    print("=== 0.2 Fiber count per experiment ===") 
    counts = df.groupby(experiment_col).size()
    print(counts.describe())
    n_singletons = int((counts == 1).sum())
    print(f"\nExperiments with only 1 fiber: {n_singletons} "
    f"({n_singletons/len(counts):.1%})")
    print(">> Singleton experiments contribute ZERO within experiment")
    print(" variance information and will distort the ICC estimate in")
    print(" step 0.8 if not handled. Report ICC both with and without")
    print(" singletons excluded, and check whether singleton only")
    print(" experiments cluster around specific proteins/devices (could")
    print(" indicate incomplete data collection rather than true design).\n")
    return counts


# ---------------------------------------------------------------
# 0.3 Missingness audit (mechanism-aware, not just a percentage)
# ---------------------------------------------------------------
def missingness_audit(df, feature_cols, group_cols=("protein_id", "device_id")):
    print("=== 0.3 Missingness audit ===")
    miss = df[feature_cols].isna().mean().sort_values(ascending=False)
    print("Missing fraction per feature:\n", miss[miss > 0])
    for g in group_cols:
        if g not in df.columns:
            continue
        print(f"\n--- Mean missingness rate by {g} (top 10) ---")
        rates = df.groupby(g)[feature_cols].apply(lambda x: x.isna().mean().mean())
        print(rates.sort_values(ascending=False).head(10))
    print("\n>> If missingness rate varies sharply across protein/device")
    print(" groups, it's likely MAR/MNAR (e.g. one device doesn't log")
    print(" humidity) rather than MCAR. Blanket mean/median imputation")
    print(" will bias those groups. Prefer model-native NaN handling")
    print(" (HGBR / LightGBM / GPBoost all support it natively) or")
    print(" group-aware imputation — never a single global fill value.\n")

# ---------------------------------------------------------------
# 0.4 Collinearity among process features (VIF)
# ---------------------------------------------------------------
def collinearity_check(df, numeric_feature_cols):
    print("=== 0.4 Collinearity / VIF ===")
    X = df[numeric_feature_cols].dropna()
    X = (X - X.mean()) / X.std() # standardize
    X = X.loc[:, X.notna().any()] # some columns have variance = 0
    #print(X.std().loc[:,X.isna().any()].to_string())
    X = sm.add_constant(X) # VIF is defined w.r.t.  an intercept
    vifs = pd.Series( [variance_inflation_factor(X.values, i) for i in range(X.shape[1])], index=X.columns,).drop("const").sort_values(ascending=False)
    print(vifs)
    print("\n>> VIF > 10 (some use >5) signals strong collinearity.  Matters")
    print(" less for GPBoost/HGBR (trees split on whichever correlated")
    print(" feature is available; mostly muddies SHAP attribution) but")
    print(" matters a lot for the LMM baseline used for effect- size")
    print(" interpretation — collinear predictors there produce unstable,")
    print(" sign-flipping coefficients. Consider dropping/ combining pairs")
    print(" with the highest VIF before fitting the LMM.\n")
    return vifs

# ---------------------------------------------------------------
# 0.5 Protein x device/protocol confound check
# ---------------------------------------------------------------
def confound_check(df, protein_col="protein_id", device_col="device_id"):
    print("=== 0.5 Protein-device confound check ===")
    ct = pd.crosstab(df[protein_col], df[device_col])
    print(ct)
    diversity = ct.astype(bool).sum(axis=1) # distinct devices per protein
    print("\nDistinct devices/protocols per protein:\n", diversity.sort_values())
    confounded = diversity[diversity == 1]
    if len(confounded):
        print(f"\n>> WARNING: {len(confounded)} protein(s) spun with only ONE")
        print(" device/protocol:", list(confounded.index))
        print(" For these, the 'protein effect' and 'device effect' are")
        print(" statistically unidentifiable — any model will attribute")
        print(" variance to whichever it happens to see first, and you")
        print(" cannot honestly claim the result is driven by SEQUENCE.")
        print(" State this explicitly as a limitation in the paper.\n")
    else:
        print("\n>> No fully confounded protein found. Sequence and process")
        print(" effects are at least partially separable in this data.\n")
    return ct
    
# ---------------------------------------------------------------
# 0.6 Protein coverage — is a LOPO fold actually viable?
# ---------------------------------------------------------------
def lopo_viability_check(df, protein_col="protein_id", experiment_col="experiment_id"):
    print("=== 0.6 LOPO fold viability ===")
    per_protein_exp = df.groupby(protein_col)[experiment_col].nunique().sort_values()
    print("Distinct experiments per protein:\n", per_protein_exp)
    per_protein_fib = df.groupby(protein_col).size().sort_values()
    print("\nFibers per protein:\n", per_protein_fib)
    thin = per_protein_exp[per_protein_exp <= 2]
    if len(thin):
        print(f"\n>> {len(thin)} protein(s) have <=2 experiments:",
        list(thin.index))
        print(" Their LOPO fold will be small and noisy. Report per-fold")
        print(" results (not just the average across all 14 folds), and")
        print(" consider excluding these from headline LOPO numbers while")
        print(" still reporting them separately for transparency.  \n")
    return per_protein_exp

# ---------------------------------------------------------------
# 0.7 Target distributions and within-experiment variability
# ---------------------------------------------------------------
def target_variability_check(df, target_cols, experiment_col="experiment_id"):
    print("=== 0.7 Target distributions & within-experiment CV ===")
    for t in target_cols:
        skew = df[t].skew()
        print(f"\n{t}: skew={skew:.2f}, min={df[t].min():.3g}, max={df[t].max():.3g}")
        if abs(skew) > 1:
            print(f" >> Notably skewed — consider a log-transform before")
            print(f" fitting Gaussian-likelihood models (GPBoost / LMM).")
        grp = df.groupby(experiment_col)[t]
        cv = (grp.std() / grp.mean()).replace([np.inf, -np.inf],
        np.nan)
        print(f" Median within-experiment CV: {cv.median():.1%} " f"(recompute this yourself rather than citing the manuscript's " f"39% figure — your preprocessing may differ)")

# ---------------------------------------------------------------
# 0.8 ICC / variance decomposition — the noise ceiling
# ---------------------------------------------------------------
def compute_icc(df, target_col, experiment_col="experiment_id"):
    print(f"=== 0.8 ICC for {target_col} ===")
    model = smf.mixedlm(f"{target_col} ~ 1", df, groups=df[experiment_col])
    result = model.fit(reml=True)
    var_between = result.cov_re.iloc[0, 0] # experiment-level variance
    var_within = result.scale # residual (fiber- level) variance
    icc = var_between / (var_between + var_within)
    print(f"Between-experiment variance: {var_between:.4f}")
    print(f"Within-experiment variance: {var_within:.4f}")
    print(f"ICC: {icc:.3f}")
    print(f">> {icc:.1%} of total variance in {target_col} is, in principle,")
    print(f" explainable by experiment-level factors (process + protein).")
    print(f" {1-icc:.1%} is irreducible fiber-to-fiber noise — no input-based")
    print(f" model can exceed R^2 ~ {icc:.2f} on this target no matter how")
    print(f" good it is. Report this number alongside every R^2 you report.\n")
    return icc
    
def run_all_icc(df, target_cols, experiment_col="experiment_id"):
    return {t: compute_icc(df, t, experiment_col) for t in target_cols}

# ---------------------------------------------------------------
# 0.9 ESM-2 embedding sanity check (only 14 proteins)
# ---------------------------------------------------------------
def embedding_sanity_check(embedding_df, protein_col="protein_id"):
    """
    embedding_df: one row per unique protein (14 rows), columns =
    embedding
    dimensions (raw 640-dim ESM-2, or your PCA-reduced version).
    """
    print("=== 0.9 Embedding structure sanity check ===")
    X = embedding_df.drop(columns=[protein_col]).values
    dist = squareform(pdist(X, metric="cosine"))
    print("Pairwise cosine distance matrix (protein x protein):")
    print(pd.DataFrame(dist, index=embedding_df[protein_col], columns=embedding_df[protein_col]).round(2))
    Z = linkage(pdist(X, metric="cosine"), method="average")
    clusters = fcluster(Z, t=3, criterion="maxclust")
    print("\n3-cluster grouping of proteins by embedding:")
    print(pd.Series(clusters, index=embedding_df[protein_col], name="cluster"))


# ---------------------------------------------------------------
# Run everything
# ---------------------------------------------------------------
ds = Dataset()
df = ds.df
df = pd.DataFrame(data=df.iloc[:,0].values, columns=['experiment_id']).join([pd.DataFrame(columns=['fiber_id'], data=[x for x in range(len(df))], dtype=np.int64), df.iloc[:,1:]])
df = df.rename(columns={
    'Spinning device': 'device_id', 
    'Protein': 'protein_id',
    'Diameter (µm)': 'diameter', 
    'Strain (mm/mm)': 'strain', 
    'Strength (MPa)': 'strength', 
    'Toughness Modulus (MJ m-3)': 'toughness', 
    'Youngs Modulus (GPa)': 'E'})

PROCESS_FEATURES = list(df.columns[2:-5])
TARGETS = list(df.columns[-5:])
NUMERIC_PROCESS_FEATURES = list(df.columns[2:-12])
CATEGORICAL_PROCESS_FEATURES = list(df.columns[13:-5])

structural_check(df)
fiber_count_distribution(df)
missingness_audit(df, PROCESS_FEATURES)
collinearity_check(df, [c for c in PROCESS_FEATURES if pd.api.types.is_numeric_dtype(df[c])])
confound_check(df)
lopo_viability_check(df)
target_variability_check(df, TARGETS)
run_all_icc(df, TARGETS)

# If you have a separate protein-level embedding table (14 rows):
embedding_df = pd.read_csv("../data/protein_embeddings.csv").rename(columns={'Protein': 'protein_id'})
embedding_sanity_check(embedding_df)

=== 0.1 Structural check ===
Rows (fibers): 2190
Unique experiments: 367
Duplicate fiber IDs: 0

Column dtypes:
 float64    16
str         8
int64       1
Name: count, dtype: int64

>> Expect ~367 experiments, ~2190 fibers. If these don't match,

>> Expect ~367 experiments, ~2190 fibers. If these don't match,
 check for duplicated rows or a schema mismatch before
 proceeding — every downstream diagnostic assumes this is right.

=== 0.2 Fiber count per experiment ===
count    367.000000
mean       5.967302
std        4.456716
min        1.000000
25%        1.000000
50%       10.000000
75%       10.000000
max       11.000000
dtype: float64

Experiments with only 1 fiber: 163 (44.4%)
>> Singleton experiments contribute ZERO within experiment
 variance information and will distort the ICC estimate in
 step 0.8 if not handled. Report ICC both with and without
 singletons excluded, and check whether singleton only
 experiments cluster around specific proteins/devices (could
 indicate incompl

/home/erik/Repositories/Explainable-AI-for-Recombinant-Spider-Silk/spidro_env/lib/python3.14/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Between-experiment variance: 0.0075
Within-experiment variance: 0.0036
ICC: 0.674
>> 67.4% of total variance in diameter is, in principle,
 explainable by experiment-level factors (process + protein).
 32.6% is irreducible fiber-to-fiber noise — no input-based
 model can exceed R^2 ~ 0.67 on this target no matter how
 good it is. Report this number alongside every R^2 you report.

=== 0.8 ICC for strain ===
Between-experiment variance: 0.0408
Within-experiment variance: 0.0221
ICC: 0.649
>> 64.9% of total variance in strain is, in principle,
 explainable by experiment-level factors (process + protein).
 35.1% is irreducible fiber-to-fiber noise — no input-based
 model can exceed R^2 ~ 0.65 on this target no matter how
 good it is. Report this number alongside every R^2 you report.

=== 0.8 ICC for strength ===


/home/erik/Repositories/Explainable-AI-for-Recombinant-Spider-Silk/spidro_env/lib/python3.14/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Between-experiment variance: 0.0057
Within-experiment variance: 0.0067
ICC: 0.460
>> 46.0% of total variance in strength is, in principle,
 explainable by experiment-level factors (process + protein).
 54.0% is irreducible fiber-to-fiber noise — no input-based
 model can exceed R^2 ~ 0.46 on this target no matter how
 good it is. Report this number alongside every R^2 you report.

=== 0.8 ICC for toughness ===
Between-experiment variance: 0.0139
Within-experiment variance: 0.0143
ICC: 0.493
>> 49.3% of total variance in toughness is, in principle,
 explainable by experiment-level factors (process + protein).
 50.7% is irreducible fiber-to-fiber noise — no input-based
 model can exceed R^2 ~ 0.49 on this target no matter how
 good it is. Report this number alongside every R^2 you report.

=== 0.8 ICC for E ===


/home/erik/Repositories/Explainable-AI-for-Recombinant-Spider-Silk/spidro_env/lib/python3.14/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


Between-experiment variance: 0.0016
Within-experiment variance: 0.0035
ICC: 0.315
>> 31.5% of total variance in E is, in principle,
 explainable by experiment-level factors (process + protein).
 68.5% is irreducible fiber-to-fiber noise — no input-based
 model can exceed R^2 ~ 0.31 on this target no matter how
 good it is. Report this number alongside every R^2 you report.

=== 0.9 Embedding structure sanity check ===
Pairwise cosine distance matrix (protein x protein):
protein_id                          NT    CT  NT-CT  NT2RepCT  A3IA  Rep1  Rep2  Rep3  Rep4  Rep5  ...  Mcherry -A3IA  \
protein_id                                                                                         ...                  
NT                                0.00  0.06   0.01      0.03  0.03  0.05  0.05  0.04  0.06  0.07  ...           0.03   
CT                                0.06  0.00   0.03      0.02  0.02  0.05  0.05  0.02  0.03  0.05  ...           0.02   
NT-CT                             0.01  0

In [ ]:
print(df[NUMERIC_PROCESS_FEATURES].dropna().std())

Capillery size (um)      12.457207
Flow rate (ul/min)       17.520212
Humidity (spinning)       0.079014
NaCl (mM)                 0.000000
Reeling speed (rpm)       7.442733
SB conc. (mM)             0.000000
SB pH                     0.000000
Temp C (spinning)         0.144164
Temperature SB            0.000000
concentration (mg/ml)     1.010461
pumppressure (bar)       30.181326
dtype: float64


In [57]:
print(df.to_string())

                 experiment_id  fiber_id  Capillery size (um)  Flow rate (ul/min)  Humidity (spinning)  NaCl (mM)  Reeling speed (rpm)  SB conc. (mM)  SB pH  Temp C (spinning)  Temperature SB  concentration (mg/ml)  pumppressure (bar) Capillery type Continous spinning Extrusion device Number of baths                           Protein       Spinning Buffer device_id  Diameter (µm)  Strain (mm/mm)  Strength (MPa)  Toughness Modulus (MJ m-3)  Youngs Modulus (GPa)
0        0.2% DMSA spin 100rpm         0                 58.0                17.0                 0.35        0.0                100.0          750.0    5.0               22.2            22.0                  326.0                 NaN          Glass                yes     Syringe pump               1                          NT2RepCT          Acetate (Na)      Hulk       0.131759        0.655213        0.202141                    0.324309              0.190678
1         0.2% DMSA spin 50rpm         1                 58.0         